# 06 — Aim 1 RNA Inventory (read-only diagnostic)

**Purpose.** Decide *before any analysis is written* whether the Aim 1 primary contrast
(pragmatic `tcab1_overall_status`: `candidate_disrupted` vs `candidate_intact`) is statistically
estimable from the RNA we actually have on disk, or whether TCAB1 status is confounded with
cohort / platform / sample type.

**Read-only.** This notebook does **not** modify the manifest, does **not** run differential
expression, and does **not** load any full expression matrix into memory (index/header
inspection only). It writes a prose report to `data/processed/notebook_06_report.md`.

**Inputs**

- `data/manifests/master_manifest.csv`
- `data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv` (index column only)
- `data/raw/target_os/TARGET-OS.star_tpm.tsv.gz` (header only — kept for future routing)

**Output**

- `data/processed/notebook_06_report.md` (this notebook is the only thing that writes it)


In [1]:
import sys, random, time
from pathlib import Path

random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ \u2014 tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import log_versions
import pandas as pd

log_versions()
print(f'Run: {RUN_TIMESTAMP}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  pyranges: 0.1.4
  openpyxl: 3.1.5
  requests: 2.33.1
  GEOparse: 2.0.4
Run: 2026-05-17T19:45:58Z


## 1 — Load manifest and subset to the Aim 1 primary contrast

We delegate arm assignment to `src.audit.classify_aim1_arms(manifest, 'pragmatic')` so this
notebook cannot drift from notebook 05. The classifier returns one of
`{'disrupted', 'intact', 'uncertain', 'excluded'}`; the primary contrast is the union of the
first two. Rows in `'uncertain'` or `'excluded'` are not part of the primary contrast.

The user spec describes the contrast as "`tcab1_overall_status` in `{candidate_disrupted, candidate_intact}`"; in this codebase those map to the `'disrupted'` and `'intact'` arms produced by the canonical classifier.

In [2]:
from src.aim1_inventory import select_pragmatic_contrast_rows

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
print(f'Manifest rows: {len(manifest)}')

contrast = select_pragmatic_contrast_rows(manifest)
print(f"Aim 1 pragmatic primary contrast rows: {len(contrast)}")
print()
print('Arm breakdown:')
print(contrast['arm'].value_counts().to_string())
print()
print('Arm \u00d7 cohort:')
print(pd.crosstab(contrast['cohort'], contrast['arm']).to_string())

Manifest rows: 115
Aim 1 pragmatic primary contrast rows: 40

Arm breakdown:
arm
intact       32
disrupted     8

Arm × cohort:
arm                disrupted  intact
cohort                              
DepMap_26Q1                2       0
Lawlor_2026_GCCRI          4       0
PCGP_2014                  2      32


## 2 — DepMap RNA membership check

Verify each DepMap `ModelID` in the contrast actually appears as a row in
`OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv`. We read only the `ModelID` column, never
the matrix body. Data type for this file is **log2(TPM+1), human protein-coding** (DepMap 26Q1
release convention).

In [3]:
from src.aim1_inventory import (
    DEPMAP_RNA_PATH, DEPMAP_RNA_DATA_TYPE,
    depmap_rna_model_ids,
)

print(f'Reading ModelID column from: {DEPMAP_RNA_PATH.name}')
depmap_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
print(f'Unique ModelIDs in DepMap RNA file: {len(depmap_models)}')
print(f'Data type: {DEPMAP_RNA_DATA_TYPE}')
print()

depmap_contrast = contrast[contrast['cohort'] == 'DepMap_26Q1'].copy()
depmap_contrast['file_backed_rna'] = depmap_contrast['sample_id'].isin(depmap_models)
print('DepMap contrast samples: file-backed RNA per row')
print(depmap_contrast[['sample_id', 'arm', 'file_backed_rna']].to_string(index=False))

missing = depmap_contrast[~depmap_contrast['file_backed_rna']]
if len(missing):
    print(f"\n!! {len(missing)} DepMap sample_id(s) from the contrast are NOT in the RNA file:")
    for _, r in missing.iterrows():
        print(f"   - {r['sample_id']}  (arm={r['arm']})")
else:
    print('\nAll DepMap contrast samples are present in the DepMap RNA matrix.')

Reading ModelID column from: OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv
Unique ModelIDs in DepMap RNA file: 1719
Data type: log2(TPM+1) protein-coding

DepMap contrast samples: file-backed RNA per row
 sample_id       arm  file_backed_rna
ACH-003171 disrupted             True
ACH-003185 disrupted             True

All DepMap contrast samples are present in the DepMap RNA matrix.


## 3 — Non-DepMap cohorts: does file-backed expression exist at all?

For Lawlor PDX, PCGP, de Nonneville, and Taglialatela: scan `data/raw/` for any expression
matrix. The registry `EXPRESSION_SOURCES` in `src/aim1_inventory.py` records what we know to
be on disk; a `None` path is a deliberate statement that no expression matrix has been
downloaded for that cohort.

If a path is registered we *also* probe header / index columns to verify that the contrast
sample IDs map cleanly. We log per-sample misses rather than silently dropping them.

In [4]:
from src.aim1_inventory import (
    EXPRESSION_SOURCES, TARGET_OS_TPM_PATH, target_os_rna_sample_ids,
)

non_depmap_cohorts = sorted(c for c in contrast['cohort'].unique() if c != 'DepMap_26Q1')
print(f'Non-DepMap cohorts in the contrast: {non_depmap_cohorts}')
print()

for cohort in non_depmap_cohorts:
    src_meta = EXPRESSION_SOURCES.get(cohort)
    sub = contrast[contrast['cohort'] == cohort]
    print(f'=== {cohort} ({len(sub)} contrast samples) ===')
    if src_meta is None:
        print('  No entry in EXPRESSION_SOURCES registry  \u2192 no expression file known for this cohort.')
        print('  These samples carry architecture/ALT labels only and have NO usable RNA in data/raw/.')
    elif src_meta.path is None:
        print(f'  Registry note: {src_meta.description}')
        print('  No expression matrix on disk \u2192 these samples have NO usable RNA in data/raw/.')
    else:
        print(f'  Registered file: {src_meta.path.name}  (data_type: {src_meta.data_type})')
        print(f'  Probe NOT implemented for this cohort \u2014 add one to src.aim1_inventory if needed.')
    print()

if TARGET_OS_TPM_PATH.exists():
    target_samples = target_os_rna_sample_ids(TARGET_OS_TPM_PATH)
    print(f'TARGET-OS reference: {TARGET_OS_TPM_PATH.name} has {len(target_samples)} sample columns')
    print('  (TARGET-OS is downloaded but not currently joined to the manifest as a cohort.)')
else:
    target_samples = set()
    print('TARGET-OS TPM file not present on disk.')

Non-DepMap cohorts in the contrast: ['Lawlor_2026_GCCRI', 'PCGP_2014']

=== Lawlor_2026_GCCRI (4 contrast samples) ===
  Registry note: Lawlor 2026 ships SV/IHC labels only; no expression matrix in data/raw/.
  No expression matrix on disk → these samples have NO usable RNA in data/raw/.

=== PCGP_2014 (34 contrast samples) ===
  Registry note: Chen 2014 PCGP cohort ships TP53 SV / ATRX / telomere-FISH tables only; no expression matrix in data/raw/.
  No expression matrix on disk → these samples have NO usable RNA in data/raw/.

TARGET-OS reference: TARGET-OS.star_tpm.tsv.gz has 88 sample columns
  (TARGET-OS is downloaded but not currently joined to the manifest as a cohort.)


## 4 — Tidy per-sample inventory

One row per contrast sample. `file_backed_rna` is `True` only when the actual expression file
contains the sample ID; `manifest_has_rna` is preserved as a separate column so we can see
where the manifest flag and the on-disk evidence disagree.

In [5]:
from src.aim1_inventory import build_inventory

inventory = build_inventory(
    contrast,
    depmap_models=depmap_models,
    target_os_samples=target_samples,
)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', 80,
                       'display.width', 200):
    print(inventory.to_string(index=False))

mismatch = inventory[inventory['manifest_has_rna'] != inventory['file_backed_rna']]
print(f'\nRows where manifest_has_rna != file_backed_rna: {len(mismatch)}')
if len(mismatch):
    print('(These are samples the manifest claims have RNA but no on-disk matrix backs the claim,')
    print(' or vice versa. Each one is reported above so nothing is silently dropped.)')

    sample_id            cohort tcab1_overall_status sample_type  manifest_has_rna  file_backed_rna                                                                                          expression_source                  data_type
   ACH-003171       DepMap_26Q1            disrupted   cell_line              True             True                                                         OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv log2(TPM+1) protein-coding
   ACH-003185       DepMap_26Q1            disrupted   cell_line              True             True                                                         OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv log2(TPM+1) protein-coding
         OS33 Lawlor_2026_GCCRI            disrupted         pdx             False            False                                   Lawlor 2026 ships SV/IHC labels only; no expression matrix in data/raw/.                    unknown
         OS37 Lawlor_2026_GCCRI            disrupted         pdx

## 5 — Cross-tabulations restricted to file-backed RNA

In [6]:
fb = inventory[inventory['file_backed_rna']].copy()
print(f'File-backed RNA samples: {len(fb)}  '
      f"(disrupted={int((fb['tcab1_overall_status']=='disrupted').sum())}, "
      f"intact={int((fb['tcab1_overall_status']=='intact').sum())})")
print()

def _xtab(df, idx_col, val_col="tcab1_overall_status"):
    if len(df) == 0:
        return pd.DataFrame()
    return pd.crosstab(df[idx_col], df[val_col])

for idx_col in ('cohort', 'sample_type', 'data_type'):
    print(f'--- {idx_col} \u00d7 tcab1_overall_status (file_backed_rna only) ---')
    ct = _xtab(fb, idx_col)
    print(ct.to_string() if not ct.empty else '(no rows)')
    print()

File-backed RNA samples: 2  (disrupted=2, intact=0)

--- cohort × tcab1_overall_status (file_backed_rna only) ---
tcab1_overall_status  disrupted
cohort                         
DepMap_26Q1                   2

--- sample_type × tcab1_overall_status (file_backed_rna only) ---
tcab1_overall_status  disrupted
sample_type                    
cell_line                     2

--- data_type × tcab1_overall_status (file_backed_rna only) ---
tcab1_overall_status        disrupted
data_type                            
log2(TPM+1) protein-coding          2



## 6 — Confounding assessment

For each arm we report the percentage breakdown by cohort, sample_type, and data_type;
and we evaluate two boolean conditions used by the decision rule:

1. Does at least one cohort contain **both arms** with file-backed RNA?
2. Does at least one cohort contain both arms with file-backed RNA **and the same `data_type`**?

In [7]:
from src.aim1_inventory import assess_confounding

assessment = assess_confounding(inventory)

print(f"File-backed RNA totals  \u2192  disrupted={assessment['n_disrupted']}, "
      f"intact={assessment['n_intact']}, total={assessment['n_total']}")
print()

def _pct_by(arm, col):
    sub = fb[fb['tcab1_overall_status'] == arm]
    if len(sub) == 0:
        return f'   (no file-backed {arm} samples)'
    counts = sub[col].value_counts()
    total = counts.sum()
    return '\n'.join(
        f'   {label:<28} {n:>3}  ({100*n/total:5.1f}%)'
        for label, n in counts.items()
    )

for arm in ('disrupted', 'intact'):
    print(f'=== arm = {arm} ===')
    for col in ('cohort', 'sample_type', 'data_type'):
        print(f' by {col}:')
        print(_pct_by(arm, col))
    print()

print('--- co-occurrence flags ---')
print(f"cohorts_with_both_arms                 : {assessment['cohorts_with_both_arms']}")
print(f"cohorts_with_both_arms_same_data_type  : {assessment['cohorts_with_both_arms_same_dtype']}")
print(f"has_within_cohort_co_occurrence              : {assessment['has_within_cohort_co_occurrence']}")
print(f"has_within_cohort_co_occurrence_same_dtype   : {assessment['has_within_cohort_co_occurrence_same_dtype']}")

File-backed RNA totals  →  disrupted=2, intact=0, total=2

=== arm = disrupted ===
 by cohort:
   DepMap_26Q1                    2  (100.0%)
 by sample_type:
   cell_line                      2  (100.0%)
 by data_type:
   log2(TPM+1) protein-coding     2  (100.0%)

=== arm = intact ===
 by cohort:
   (no file-backed intact samples)
 by sample_type:
   (no file-backed intact samples)
 by data_type:
   (no file-backed intact samples)

--- co-occurrence flags ---
cohorts_with_both_arms                 : []
cohorts_with_both_arms_same_data_type  : []
has_within_cohort_co_occurrence              : False
has_within_cohort_co_occurrence_same_dtype   : False


## 7 — Apply the design decision rule

Three categories, applied in this order:

1. **Within-cohort viable** — \u22651 (cohort \u00d7 data_type) pair has both arms with \u22653 samples per arm.
2. **Pooled-exploratory only** — no single cohort has both arms, but each arm spans \u22652 cohorts.
3. **Not estimable as defined** — anything else (an arm has zero file-backed RNA, or disrupted
   and intact come from disjoint single cohorts).

In [8]:
from src.aim1_inventory import decide_aim1_design

verdict = decide_aim1_design(inventory, assessment)
print(f'VERDICT: {verdict["category"]}')
print()
print(verdict['justification'])

VERDICT: not_estimable

File-backed RNA is available for 2 disrupted sample(s) (from cohort(s) DepMap_26Q1) and 0 intact sample(s) (from cohort(s) (none)). TCAB1 status is perfectly or near-perfectly confounded with cohort/platform/sample_type, so the Aim 1 primary contrast as currently defined is not statistically estimable. The contrast must be redefined (e.g. require RNA-backed cohorts on both sides) or restricted (e.g. drop arms without file-backed RNA) before any DE analysis can proceed.


## 8 — Write `data/processed/notebook_06_report.md`

In [9]:
REPORT_PATH = ROOT / 'data' / 'processed' / 'notebook_06_report.md'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

VERDICT_LABEL = {
    'within_cohort_viable':     'WITHIN-COHORT VIABLE',
    'pooled_exploratory_only':  'POOLED-EXPLORATORY ONLY',
    'not_estimable':            'NOT ESTIMABLE AS DEFINED',
}[verdict['category']]

def _md_table(df, index_label):
    if df.empty:
        return '(no rows)\n'
    cols = list(df.columns)
    header = '| ' + index_label + ' | ' + ' | '.join(str(c) for c in cols) + ' |\n'
    sep    = '|' + '|'.join(['---'] * (len(cols) + 1)) + '|\n'
    body   = ''.join(
        '| ' + str(idx) + ' | ' + ' | '.join(str(v) for v in row) + ' |\n'
        for idx, row in df.iterrows()
    )
    return header + sep + body

lines: list[str] = []
lines.append('# Notebook 06 \u2014 Aim 1 RNA Inventory\n')
lines.append(f'_Generated: {RUN_TIMESTAMP}_\n')
lines.append('')
lines.append('Read-only diagnostic. Determines whether the Aim 1 primary contrast '
             '(pragmatic `tcab1_overall_status` \u2014 `candidate_disrupted` vs `candidate_intact`) '
             'is statistically estimable from the RNA available on disk, or whether TCAB1 '
             'status is confounded with cohort / platform / sample type.\n')
lines.append('')
lines.append('## Inputs\n')
lines.append(f'- Manifest: `{MANIFEST_PATH.relative_to(ROOT)}` ({len(manifest)} rows)')
lines.append(f'- DepMap RNA: `{DEPMAP_RNA_PATH.relative_to(ROOT)}` ({len(depmap_models)} unique ModelIDs)')
lines.append(f'- TARGET-OS TPM: `{TARGET_OS_TPM_PATH.relative_to(ROOT)}` '
             f'({len(target_samples)} sample columns; not joined to manifest as a cohort)')
lines.append('')
lines.append('## Primary contrast composition\n')
lines.append(f'Total contrast rows: **{len(contrast)}** '
             f"(disrupted={int((contrast['arm']=='disrupted').sum())}, "
             f"intact={int((contrast['arm']=='intact').sum())}).\n")
lines.append('Arm \u00d7 cohort:\n')
lines.append('```')
lines.append(pd.crosstab(contrast['cohort'], contrast['arm']).to_string())
lines.append('```\n')

lines.append('## Per-sample RNA-backing inventory\n')
lines.append('| sample_id | cohort | arm | sample_type | manifest_has_rna | file_backed_rna | expression_source | data_type |')
lines.append('|---|---|---|---|---|---|---|---|')
for _, r in inventory.iterrows():
    lines.append('| ' + ' | '.join(str(r[c]) for c in [
        'sample_id', 'cohort', 'tcab1_overall_status', 'sample_type',
        'manifest_has_rna', 'file_backed_rna', 'expression_source', 'data_type',
    ]) + ' |')
lines.append('')

lines.append('## Cross-tabulations (file-backed RNA only)\n')
for idx_col in ('cohort', 'sample_type', 'data_type'):
    lines.append(f'### `{idx_col}` \u00d7 arm\n')
    ct = pd.crosstab(fb[idx_col], fb['tcab1_overall_status']) if len(fb) else pd.DataFrame()
    lines.append(_md_table(ct, idx_col))

lines.append('## Confounding assessment (file-backed RNA only)\n')
lines.append(f"- File-backed totals: **disrupted={assessment['n_disrupted']}**, "
             f"**intact={assessment['n_intact']}**, total={assessment['n_total']}.")
for arm in ('disrupted', 'intact'):
    sub = fb[fb['tcab1_overall_status'] == arm]
    lines.append(f"- arm `{arm}` ({len(sub)} sample(s)):")
    if len(sub) == 0:
        lines.append('    - (no file-backed samples)')
        continue
    for col in ('cohort', 'sample_type', 'data_type'):
        counts = sub[col].value_counts()
        total = counts.sum()
        bits = '; '.join(f'{lab}: {n} ({100*n/total:.0f}%)' for lab, n in counts.items())
        lines.append(f'    - by {col}: {bits}')
lines.append('')
lines.append(f"- Cohorts with both arms file-backed: `{assessment['cohorts_with_both_arms']}`")
lines.append(f"- Cohorts with both arms file-backed under the same data_type: "
             f"`{assessment['cohorts_with_both_arms_same_dtype']}`")
lines.append(f"- `has_within_cohort_co_occurrence`: **{assessment['has_within_cohort_co_occurrence']}**")
lines.append(f"- `has_within_cohort_co_occurrence_same_dtype`: "
             f"**{assessment['has_within_cohort_co_occurrence_same_dtype']}**")
lines.append('')

lines.append('## AIM 1 DESIGN VERDICT\n')
lines.append(f'**Category: {VERDICT_LABEL}**\n')
lines.append('Supporting counts:\n')
lines.append(f'- File-backed RNA disrupted samples: {assessment["n_disrupted"]}')
lines.append(f'- File-backed RNA intact samples: {assessment["n_intact"]}')
lines.append(f'- Cohorts contributing to disrupted arm (file-backed): '
             f"{assessment['cohorts_per_arm']['disrupted']}")
lines.append(f'- Cohorts contributing to intact arm (file-backed): '
             f"{assessment['cohorts_per_arm']['intact']}")
lines.append(f'- Cohorts carrying both arms (same data_type): '
             f"{assessment['cohorts_with_both_arms_same_dtype']}")
lines.append('')
lines.append('Justification:\n')
lines.append(f'> {verdict["justification"]}')
lines.append('')

REPORT_PATH.write_text('\n'.join(lines))
print(f'Wrote {REPORT_PATH.relative_to(ROOT)}  ({REPORT_PATH.stat().st_size} bytes)')
print(f'Verdict: {VERDICT_LABEL}')

Wrote data/processed/notebook_06_report.md  (9842 bytes)
Verdict: NOT ESTIMABLE AS DEFINED


## 9 — Idempotence check

Re-run the report-generation logic in memory and confirm the bytes match what we just wrote.

In [10]:
import hashlib

first_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()
REPORT_PATH.write_text('\n'.join(lines))  # second write of same string
second_hash = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()

print(f'first  sha256: {first_hash}')
print(f'second sha256: {second_hash}')
assert first_hash == second_hash, 'Notebook 06 is not idempotent \u2014 investigate.'
print('Idempotent: re-writing produces identical bytes.')

first  sha256: 38c2853cae872f027ea5d1197e470f3763e55426c93c2b65853da45018aecb69
second sha256: 38c2853cae872f027ea5d1197e470f3763e55426c93c2b65853da45018aecb69
Idempotent: re-writing produces identical bytes.
